# 🏙️ Traffy Fondue Machine Learning: Full Pipeline & Model Training
### ระบบ AI พยากรณ์ระยะเวลาซ่อมแซมปัญหาเมือง กทม. ด้วย LightGBM (11 Features) & Specialized Sub-models

สมุดโค้ด (Notebook) นี้แสดงการทำงานครบวงจรตั้งแต่:
1. **Data Loading:** โหลดข้อมูลจริงที่คลีนแล้วจาก `data/processed/`
2. **Feature Matrix:** วิเคราะห์ 11 Features หน้างาน
3. **Model Training (Part 1):** ฝึกสอน Single Global LightGBM Model พร้อมพล็อต Feature Importance
4. **Model Training (Part 2):** ฝึกสอน Specialized Sub-models (แยก 5 ฝ่ายหลัก + General Fallback)
5. **Head-to-Head Comparison:** วัดผลเปรียบเทียบความแม่นยำบน Holdout Test Set ปี 2024 (49,010 เคส)
6. **Per-Department Analysis:** เจาะลึกผลลัพธ์รายหน่วยงาน
7. **Actionable Decision Layer:** คำนวณ Public Impact Score และจัดกลุ่ม 4 Quadrants Matrix
8. **Interactive Prediction Demo:** ทดลองใส่ข้อมูลรับเรื่องจริง


In [ ]:
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, median_absolute_error

# กำหนด Path หลักของโปรเจกต์
BASE_DIR = os.getcwd()
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
MODELS_DIR = os.path.join(BASE_DIR, "models")
SPECIALIZED_DIR = os.path.join(MODELS_DIR, "specialized")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(SPECIALIZED_DIR, exist_ok=True)

print("Environment Setup Complete!")
print(f"Base Directory: {BASE_DIR}")
print(f"Data Directory: {PROCESSED_DIR}")


---
## 1. การโหลดชุดข้อมูลที่คลีนสมบูรณ์แล้ว (Cleaned Datasets)
ดึงข้อมูลจากโฟลเดอร์ `data/processed/` (รูปแบบ Parquet โหลดข้อมูลกว่า 275,000 แถวได้ในเวลา < 0.5 วินาที)


In [ ]:
train_path = os.path.join(PROCESSED_DIR, "train_cleaned.parquet")
val_path = os.path.join(PROCESSED_DIR, "val_cleaned.parquet")
test_path = os.path.join(PROCESSED_DIR, "test_cleaned.parquet")

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)
test_df = pd.read_parquet(test_path)

print(f"📊 Train Set (ปี 2023 ทั้งปี)         : {len(train_df):,} แถว")
print(f"📊 Validation Set (ต้นปี 2024: ม.ค.-มี.ค.): {len(val_df):,} แถว")
print(f"📊 Test Set (กลางปี 2024: เม.ย.-มิ.ย.)   : {len(test_df):,} แถว")
print(f"--> รวมทั้งหมด                          : {len(train_df) + len(val_df) + len(test_df):,} แถว")

# ตัวอย่างข้อมูล 5 แถวแรก
train_df[['ticket_id', 'district', 'main_type', 'sub_category', 'predicted_dept', 'severity', 'duration_days']].head()


---
## 2. ตัวแปรต้น 11 Features หน้างาน (The 11 Front-End Features)
ตัวแปรที่ระบบทราบได้ ณ วินาทีแรกที่ประชาชนกดแจ้งเรื่อง โดยแบ่งเป็น Categorical, Numerical และ Binary


In [ ]:
feature_cols = [
    'district', 'main_type', 'sub_category', 'predicted_dept',
    'severity', 'comment_len', 'day_of_week', 'is_weekend',
    'month', 'is_rainy_season', 'hour'
]
cat_cols = ['district', 'main_type', 'sub_category', 'predicted_dept']

print("11 Features หน้างาน (ตัวแปร X):")
for i, col in enumerate(feature_cols, 1):
    c_type = "Categorical" if col in cat_cols else ("Binary" if "is_" in col else "Numerical")
    print(f"  {i:2d}. {col:<18} | ประเภท: {c_type:<12} | ค่าไม่ซ้ำ: {train_df[col].nunique()}")

# สถิติ Target: duration_days (ระยะเวลาซ่อมเสร็จจริง - วัน)
print("
สถิติของ Target (duration_days):")
print(train_df['duration_days'].describe(percentiles=[0.25, 0.50, 0.75, 0.90]))


---
## 3. [TRAINING] ฝึกสอนโมเดลเดี่ยว (Single Global LightGBM Model)
* **Objective:** `regression_l1` (MAE Loss - มุ่งเน้นมัธยฐาน ทนต่อ Outliers)
* **Target:** $\log(1 + 	ext{duration\_days})$
* **Categorical Handling:** ส่ง Categorical Features แบบ Native ให้ LightGBM แตกกิ่งโดยตรง


In [ ]:
# จัดเตรียม Categorical Type ให้ตรงกันระหว่าง Train, Val, Test
cat_mappings = {}
train_single = train_df.copy()
val_single = val_df.copy()
test_single = test_df.copy()

for c in cat_cols:
    train_cats = sorted(list(train_single[c].dropna().unique()))
    train_single[c] = pd.Categorical(train_single[c], categories=train_cats)
    val_single[c] = pd.Categorical(val_single[c], categories=train_cats)
    test_single[c] = pd.Categorical(test_single[c], categories=train_cats)
    cat_mappings[c] = [str(x) for x in train_cats]

# บันทึก categories.json
with open(os.path.join(MODELS_DIR, "categories.json"), "w", encoding="utf-8") as f:
    json.dump(cat_mappings, f, ensure_ascii=False, indent=2)

X_train = train_single[feature_cols]
y_train = np.log1p(train_single['duration_days'])

X_val = val_single[feature_cols]
y_val = np.log1p(val_single['duration_days'])

train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data, categorical_feature=cat_cols)

params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 45,
    'min_child_samples': 40,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'cat_smooth': 10.0,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

print("กำลังฝึกสอน Single LightGBM Model (800 rounds, early stopping 35)...")
start_t = time.time()
single_model = lgb.train(
    params,
    train_data,
    num_boost_round=800,
    valid_sets=[train_data, val_data],
    callbacks=[lgb.early_stopping(stopping_rounds=35, verbose=False)]
)
print(f"--> ฝึกสอนเสร็จสิ้นในเวลา: {time.time() - start_t:.2f} วินาที (Best Iteration: {single_model.best_iteration})")

# บันทึกโมเดล
single_model_path = os.path.join(MODELS_DIR, "lightgbm_traffy_real.txt")
single_model.save_model(single_model_path)
print(f"บันทึกโมเดลไว้ที่: {single_model_path}")


---
### Feature Importance: ตัวแปรไหนส่งผลต่อระยะเวลาซ่อมแซมมากที่สุด?


In [ ]:
# คำนวณ Feature Importance จาก Gain
importance_gain = single_model.feature_importance(importance_type='gain')
imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Gain': importance_gain
}).sort_values(by='Gain', ascending=True)

imp_df['Gain_Pct'] = (imp_df['Gain'] / imp_df['Gain'].sum()) * 100

print(imp_df.sort_values(by='Gain_Pct', ascending=False).to_string(index=False))

# พล็อตแท่ง Feature Importance
plt.figure(figsize=(9, 5))
plt.barh(imp_df['Feature'], imp_df['Gain_Pct'], color='#2b5c8f')
plt.xlabel("ความสำคัญต่อการตัดสินใจ (Gain %)")
plt.title("LightGBM Feature Importance (Top Predictors of Duration)")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


---
## 4. [TRAINING] ฝึกสอน Specialized Sub-models (Mixture of Experts)
ฝึกสอนโมเดลเฉพาะทางของ **5 ฝ่ายหลัก** + **1 General Fallback** เพื่อให้แต่ละโมเดลเรียนรู้ลักษณะงานเฉพาะด้านของตัวเอง:
1. `model_yotha.txt`: ฝ่ายโยธา (num_leaves=63 สำหรับงานก่อสร้างที่ซับซ้อน)
2. `model_cleanliness.txt`: ฝ่ายรักษาความสะอาดฯ
3. `model_thetsakit.txt`: ฝ่ายเทศกิจ
4. `model_environment.txt`: ฝ่ายสิ่งแวดล้อมฯ
5. `model_drainage.txt`: สำนักการระบายน้ำ
6. `model_general.txt`: General Fallback Model


In [ ]:
TARGET_DEPTS = [
    'ฝ่ายโยธา',
    'ฝ่ายรักษาความสะอาดฯ',
    'ฝ่ายเทศกิจ',
    'ฝ่ายสิ่งแวดล้อมฯ',
    'สำนักการระบายน้ำ'
]

DEPT_SLUG_MAP = {
    'ฝ่ายโยธา': 'yotha',
    'ฝ่ายรักษาความสะอาดฯ': 'cleanliness',
    'ฝ่ายเทศกิจ': 'thetsakit',
    'ฝ่ายสิ่งแวดล้อมฯ': 'environment',
    'สำนักการระบายน้ำ': 'drainage',
}

def train_specialized_submodel(dept_name, train_sub, val_sub):
    slug = DEPT_SLUG_MAP.get(dept_name, dept_name)
    sub_cat_cols = ['district', 'main_type', 'sub_category']
    sub_feat_cols = [
        'district', 'main_type', 'sub_category',
        'severity', 'comment_len', 'day_of_week',
        'is_weekend', 'month', 'is_rainy_season', 'hour'
    ]
    
    t_df = train_sub.copy()
    v_df = val_sub.copy()
    
    sub_cats = {}
    for c in sub_cat_cols:
        cats = sorted(list(t_df[c].dropna().unique()))
        t_df[c] = pd.Categorical(t_df[c], categories=cats)
        v_df[c] = pd.Categorical(v_df[c], categories=cats)
        sub_cats[c] = [str(x) for x in cats]
        
    with open(os.path.join(SPECIALIZED_DIR, f"categories_{slug}.json"), "w", encoding="utf-8") as f:
        json.dump(sub_cats, f, ensure_ascii=False, indent=2)
        
    X_tr = t_df[sub_feat_cols]
    y_tr = np.log1p(t_df['duration_days'])
    X_va = v_df[sub_feat_cols]
    y_va = np.log1p(v_df['duration_days'])
    
    tr_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=sub_cat_cols)
    va_data = lgb.Dataset(X_va, label=y_va, reference=tr_data, categorical_feature=sub_cat_cols)
    
    n_leaves = 63 if dept_name in ['ฝ่ายโยธา', 'general'] else 31
    p = {
        'objective': 'regression_l1',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': n_leaves,
        'min_child_samples': 30,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'cat_smooth': 10.0,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    m = lgb.train(
        p, tr_data, num_boost_round=800,
        valid_sets=[tr_data, va_data],
        callbacks=[lgb.early_stopping(stopping_rounds=35, verbose=False)]
    )
    
    m_path = os.path.join(SPECIALIZED_DIR, f"model_{slug}.txt")
    m.save_model(m_path)
    return m, sub_cats

print("🚀 เริ่มต้นฝึกสอน Specialized Sub-models ทั้ง 6 โมเดล...")
specialist_models = {}
specialist_cats = {}

# เทรน 5 ฝ่ายหลัก
for dept in TARGET_DEPTS:
    tr_sub = train_df[train_df['predicted_dept'] == dept]
    va_sub = val_df[val_df['predicted_dept'] == dept]
    print(f"  [+] กำลังฝึกสอน: {dept} (Train: {len(tr_sub):,} | Val: {len(va_sub):,})")
    m, c = train_specialized_submodel(dept, tr_sub, va_sub)
    specialist_models[dept] = m
    specialist_cats[dept] = c

# เทรน General Fallback บนข้อมูลทั้งหมด
print("  [+] กำลังฝึกสอน: General Fallback Model...")
m_gen, c_gen = train_specialized_submodel("general", train_df, val_df)
specialist_models["general"] = m_gen
specialist_cats["general"] = c_gen

with open(os.path.join(SPECIALIZED_DIR, "dept_map.json"), "w", encoding="utf-8") as f:
    json.dump(DEPT_SLUG_MAP, f, ensure_ascii=False, indent=2)

print("--> [สำเร็จ] ฝึกสอนและบันทึกโมเดลเฉพาะทางครบถ้วนทั้ง 6 โมเดลแล้ว!")


---
## 5. การแข่งขันวัดผลเปรียบเทียบ (Head-to-Head Evaluation)
นำ **Test Set 49,010 เคสจริง (กลางปี 2024)** มาประเมินแบบ Head-to-Head ระหว่าง Single Model vs Specialized Sub-models


In [ ]:
from src.specialized_predictor import SpecializedBMAPredictor

# เรียกใช้ SpecializedBMAPredictor
predictor = SpecializedBMAPredictor(models_dir=SPECIALIZED_DIR)

# 1. ทำนายด้วย Single Model
test_eval_df = test_df.copy()
for c in cat_cols:
    test_eval_df[c] = pd.Categorical(test_eval_df[c], categories=cat_mappings.get(c, []))

single_preds_log = single_model.predict(test_eval_df[feature_cols])
test_eval_df['single_pred_days'] = np.expm1(single_preds_log)

# 2. ทำนายด้วย Specialized Sub-models ผ่าน Router
test_eval_df = predictor.predict_df(test_eval_df)

# คำนวณตัวชี้วัดภาพรวม
actual = test_eval_df['duration_days'].values
s_mae = mean_absolute_error(actual, test_eval_df['single_pred_days'])
s_med = median_absolute_error(actual, test_eval_df['single_pred_days'])
p_mae = mean_absolute_error(actual, test_eval_df['specialized_pred_days'])
p_med = median_absolute_error(actual, test_eval_df['specialized_pred_days'])

s_acc3 = np.mean(np.abs(actual - test_eval_df['single_pred_days']) <= 3.0) * 100
p_acc3 = np.mean(np.abs(actual - test_eval_df['specialized_pred_days']) <= 3.0) * 100

print("=" * 65)
print("🏆 สรุปผลการแข่งขันภาพรวม (Overall Benchmark 49,010 เคส)")
print("=" * 65)
print(f"ตัวชี้วัด                      โมเดลเดี่ยว (Single)   โมเดลเฉพาะทาง (Specialized)")
print(f"1. MAE (คลาดเคลื่อนเฉลี่ย)    :    {s_mae:5.2f} วัน           {p_mae:5.2f} วัน")
print(f"2. Median AE (มัธยฐานคลาดเคลื่อน):  {s_med:5.2f} วัน           {p_med:5.2f} วัน")
print(f"3. ความแม่นยำในกรอบ ±3 วัน    :   {s_acc3:5.2f}%           {p_acc3:5.2f}%")
print("=" * 65)


---
## 6. เจาะลึกผลลัพธ์รายหน่วยงาน (Per-Department Breakdown)
ตารางเปรียบเทียบความแม่นยำแยกตามฝ่ายที่รับผิดชอบ


In [ ]:
dept_stats = []
for d in TARGET_DEPTS:
    sub = test_eval_df[test_eval_df['predicted_dept'] == d]
    if len(sub) == 0:
        continue
    act = sub['duration_days'].values
    sm = mean_absolute_error(act, sub['single_pred_days'])
    smed = median_absolute_error(act, sub['single_pred_days'])
    pm = mean_absolute_error(act, sub['specialized_pred_days'])
    pmed = median_absolute_error(act, sub['specialized_pred_days'])
    impr = ((sm - pm) / sm) * 100
    
    dept_stats.append({
        'หน่วยงาน': d,
        'จำนวนเคส': len(sub),
        'Single MAE': round(sm, 2),
        'Spec MAE': round(pm, 2),
        'Single MedAE': round(smed, 2),
        'Spec MedAE': round(pmed, 2),
        'MAE Improvement (%)': round(impr, 2)
    })

comparison_table = pd.DataFrame(dept_stats)
comparison_table


---
## 7. Actionable Decision Layer: จัดคิวสั่งการ กทม. (4 Quadrants Matrix)
แปลงผลทำนายระยะเวลา + ความรุนแรง (Severity) + ความหนาแน่นประชากร (Density Factor) เป็น Action สั่งการทันที


In [ ]:
from src.decision_layer import BMADecisionLayer

decision_layer = BMADecisionLayer()

# ประเมินตัวอย่าง 1,000 เคส
sample_eval = test_eval_df.head(1000).copy()
triage_df = decision_layer.evaluate_dataframe(sample_eval, duration_col='specialized_pred_days')

print("สัดส่วนปัญหาในแต่ละ Quadrant (1,000 เคสตัวอย่าง):")
print(triage_df['triage_quadrant'].value_counts())

print("
ตัวอย่างเคสวิกฤตเร่งด่วน (Critical Urgent - ต้องจบใน 24 ชม.):")
urgent = triage_df[triage_df['triage_quadrant'] == 'Critical Urgent']
urgent[['district', 'main_type', 'sub_category', 'severity', 'specialized_pred_days', 'public_impact_score', 'bma_action']].head(3)


---
## 8. Interactive Prediction Demo: ทดลองรับเรื่องจริง
คุณสามารถปรับเปลี่ยนข้อมูลด้านล่างเพื่อทดสอบจำลองการส่งเรื่องเข้าสู่ระบบ AI


In [ ]:
def simulate_ticket(district, main_type, sub_category, predicted_dept, severity, comment):
    case = {
        'district': district,
        'main_type': main_type,
        'sub_category': sub_category,
        'severity': severity,
        'comment_len': len(comment),
        'day_of_week': 2,        # พุธ
        'is_weekend': 0,
        'month': 6,              # มิ.ย.
        'is_rainy_season': 1,    # ฝน
        'hour': 9
    }
    
    res = predictor.predict_single(case, predicted_dept=predicted_dept)
    pred_days = res['predicted_days']
    triage = decision_layer.triage_ticket(district, severity, pred_days)
    
    print("=" * 60)
    print("📋 ผลการวิเคราะห์เรื่องร้องเรียน Traffy Fondue")
    print("=" * 60)
    print(f"พื้นที่แจ้งเหตุ : เขต{district}")
    print(f"ข้อความแจ้งเหตุ : '{comment}'")
    print(f"ฝ่ายรับผิดชอบ  : {predicted_dept}")
    print(f"โมเดลที่เลือกใช้ : {res['model_used']} (Slug: {res['model_slug']})")
    print("-" * 60)
    print(f"⏱️ เวลาคาดการณ์  : {pred_days} วัน (~{pred_days*24:.1f} ชั่วโมง)")
    print(f"🚨 Triage กลุ่ม : {triage['quadrant']}")
    print(f"💥 Impact Score: {triage['impact_score']}")
    print(f"📢 คำสั่งการ กทม.: {triage['action']}")
    print("=" * 60)

# ทดสอบเคสจำลอง
simulate_ticket(
    district="จตุจักร",
    main_type="ขยะ",
    sub_category="ขยะตกค้างส่งกลิ่นเหม็น",
    predicted_dept="ฝ่ายรักษาความสะอาดฯ",
    severity=3,
    comment="ถังขยะหน้าปากซอยล้นมา 2 วันแล้ว ส่งกลิ่นเหม็นรบกวนคนเดินผ่านไปมา"
)
